# 2-5. RAG 평가: RAGAS

<!-- optional --> 수업 시간이 부족하면 생략 가능합니다 (Day2 S5에서 다시 등장).

## 학습 목표
- RAGAS 핵심 지표 **faithfulness · answer_relevancy · context_precision · context_recall** 각각이 **어떻게 계산**되고 **점수 구간별로 무엇을 의미**하는지 설명할 수 있다.
- 지표가 낮을 때 **retriever 문제인지 generator 문제인지** 진단하는 플로우를 이해한다.
- 이전 노트북의 캡스톤 베이스라인을 직접 평가하고 결과를 해석한다.

## 핵심 키워드
`RAGAS` `faithfulness` `answer_relevancy` `context_precision` `context_recall` `LLM-as-judge`

> 🔒 RAGAS 지표 계산은 **LLM을 judge로 사용**한다. 폐쇄망에서는 동일한 로컬 LLM(Ollama)을 judge로 지정해 완전 오프라인으로 평가 가능.

In [1]:
import sys; sys.path.insert(0, '../..')
from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

☁️ OpenAI | model=openai/gpt-4o-mini


## 1. 핵심 지표 4가지

RAGAS는 RAG 파이프라인을 **retrieval 품질 2개 + generation 품질 2개**로 쪼개 평가한다. 모든 지표는 **[0, 1] 범위**이며 값이 클수록 좋다. 내부적으로 LLM judge가 문장 단위 판정을 수행하므로, judge 모델의 품질이 지표의 상한이 된다.

| 레이어 | 지표 | 측정 대상 | 문제가 있다면? |
|---|---|---|---|
| Generator | **faithfulness** | 답변 ↔ 컨텍스트 | LLM이 문서에 없는 내용을 지어냄 (환각) |
| Generator | **answer_relevancy** | 질문 ↔ 답변 | 답변이 질문의 의도에서 벗어남 |
| Retriever | **context_precision** | 검색결과 순위 | 관련 없는 chunk가 상위에 있음 (노이즈) |
| Retriever | **context_recall** | 정답 ↔ 검색결과 | 정답에 필요한 정보가 누락됨 |

---

### 1) Faithfulness (충실도) — 답변은 컨텍스트로 뒷받침되는가

**계산 방식**
1. LLM judge가 답변을 **원자적 주장(claim)** 으로 분해. 예: "보험 해지는 14일 내 서면으로 가능" → ① 14일 내 가능, ② 서면이어야 함.
2. 각 claim이 제공된 컨텍스트로 **추론 가능한지** LLM이 이진 판정.
3. `faithfulness = (뒷받침되는 claim 수) / (전체 claim 수)`

**정량적 해석**
- **0.9 이상** — 거의 모든 주장이 문서 근거에서 유래. 프로덕션 목표선.
- **0.7 ~ 0.9** — LLM의 사전 지식이 일부 섞임. "컨텍스트에 없으면 모른다고 답하라" 같은 프롬프트 제약 강화 필요.
- **0.7 미만** — 환각 비중이 큼. 리트리버 품질(recall) 또는 프롬프트 재설계 필요.

### 2) Answer Relevancy (답변 관련성) — 질문에 직접 대답하는가

**계산 방식**
1. LLM judge가 답변을 보고 **"이 답변을 낳았을 법한 가상의 질문"** 을 n개(보통 3개) 역생성.
2. 역생성된 질문들을 임베딩하여 **원래 질문과의 코사인 유사도** 계산.
3. `answer_relevancy = mean(cosine_sim(원본 질문, 역질문_i))`

**정량적 해석**
- **0.8 이상** — 답변이 질문 의도를 그대로 반영.
- **0.5 ~ 0.8** — 핵심은 잡았으나 불필요한 부연이 많거나 초점이 어긋남.
- **0.5 미만** — 질문과 다른 주제로 응답. 문서 나열만 하고 질문에 답하지 않는 프롬프트에서 자주 발생.

> ⚠️ **주의**: 답변이 사실적으로 정확하더라도 질문-응답 형식에서 벗어나면 점수가 낮게 나온다. "모르겠습니다"처럼 회피하는 답변, 장황한 부연 설명도 감점 대상.

### 3) Context Precision (컨텍스트 정확도) — 상위권에 정답 문서가 있는가

**계산 방식**
1. 각 컨텍스트 chunk에 대해, `ground_truth`와 관련 있는지 LLM이 이진 판정 (v_k ∈ {0,1}).
2. 순위 k에서의 Precision@k를 계산하여 **앞쪽 가중 평균** (MAP 스타일):

   `context_precision ≈ Σ_k (Precision@k × v_k) / (전체 relevant chunk 수)`

3. 즉, **관련 문서가 top-1에 있으면 만점, 뒤쪽에 있을수록 감점**.

**정량적 해석**
- **0.9 이상** — 최상위(top-1, top-2)에 정답 근거가 위치. 리트리버 양호.
- **0.5 ~ 0.9** — 관련 chunk는 있으나 노이즈와 섞여 있음. **reranker** 도입 검토.
- **0.5 미만** — 상위권에 관련성 없는 문서가 많음. 쿼리 재작성, hybrid 검색(BM25+dense), reranker 등이 필요.

### 4) Context Recall (컨텍스트 재현율) — 정답을 만들기에 정보가 충분한가

**계산 방식**
1. LLM judge가 `ground_truth`를 문장 단위로 분해.
2. 각 문장이 **컨텍스트 내용만으로 설명 가능한지** 판정.
3. `context_recall = (컨텍스트로 커버되는 gt 문장 수) / (gt 전체 문장 수)`

**정량적 해석**
- **0.9 이상** — 정답에 필요한 정보가 거의 다 검색됨.
- **0.5 ~ 0.9** — 일부 핵심 내용 누락. top-k 확대, 청킹 크기·overlap 조정 검토.
- **0.5 미만** — 정답 근거가 컨텍스트에 거의 없음. 인덱싱 실패 / 쿼리-문서 vocabulary mismatch / chunk 크기 부적절 — **여기서부터 고쳐야** 다른 지표도 의미가 생긴다.

---

### 📌 진단 플로우 — 점수가 낮을 때 어디부터 볼 것인가

```
context_recall 낮음  → 리트리버 커버리지 문제 (청킹·임베딩·top-k 조정)
    │ (recall은 OK인데)
    ▼
context_precision 낮음 → 랭킹 품질 문제 (reranker, hybrid search)
    │ (검색은 OK인데)
    ▼
faithfulness 낮음    → LLM 환각 (프롬프트에 "근거만 사용" 제약 강화)
    │ (충실도도 OK인데)
    ▼
answer_relevancy 낮음 → 프롬프트가 질문-응답 형식을 지키지 못함
```

> 💡 **레이어 분리가 핵심**: `faithfulness` / `answer_relevancy`는 **generator**(LLM + 프롬프트), `context_precision` / `context_recall`은 **retriever** 문제다. 원인 레이어를 혼동하면 엉뚱한 곳을 고치게 된다.
>
> 또한 `context_recall`이 0이면 이론적으로 `faithfulness`가 높아도 의미가 없다(정답과 무관한 내용을 충실하게 말한 것뿐). **recall부터 확보 → precision → generation** 순으로 개선하는 것이 정석.

## 2. 캐프스톤 베이스라인 로드

In [2]:
import json
from pathlib import Path

baseline_path = Path('./_capstone_baseline.json')
if not baseline_path.exists():
    raise FileNotFoundError('04_rag_pipeline_lcel.ipynb 를 먼저 실행해주세요.')

baseline = json.loads(baseline_path.read_text(encoding='utf-8'))

# 평가용 ground truth — 수업용으로 강사가 사전 작성해둔 정답
GROUND_TRUTH = [
    '14일 이내에 서면으로 철회할 수 있으나, 이미 서비스가 제공되었거나 영업일 기준 3일 이내 증정이 끝난 경우 제한된다.',
    '이용자는 즉시 고객센터 또는 가장 가까운 영업점에 신고해야 하며, 선의무과실이 없다면 신고 이전 사고도 보상책임이 금융회사에 있다.',
    '수수료 인상 적용일 30일 전에 공지해야 한다.',
    '거래일로부터 1년 이내에 이의신청 가능하고, 금융회사는 접수일로부터 15영업일 이내 결과를 통지한다.',
    '원칙적으로 24시간 이용 가능하나, 시스템 점검·장애·운영상 필요에 따라 일시 제한될 수 있으며 사전 공지된다.',
]

for i, row in enumerate(baseline):
    row['ground_truth'] = GROUND_TRUTH[i]

print(f'평가 대상 샘플 {len(baseline)}개 준비 완료')

평가 대상 샘플 5개 준비 완료


## 3. RAGAS 평가 실행

In [3]:
from datasets import Dataset

dataset = Dataset.from_list([
    {
        'question': r['question'],
        'answer': r['answer'],
        'contexts': r['contexts'],
        'ground_truth': r['ground_truth'],
    }
    for r in baseline
])
print(dataset)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})


In [4]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# judge LLM과 embedding을 common.get_chat_model / get_embeddings 로 주입 → 오픈/오프라인 모두 동작
judge_llm = LangchainLLMWrapper(get_chat_model(temperature=0))
judge_emb = LangchainEmbeddingsWrapper(get_embeddings())

result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
    embeddings=judge_emb,
)
print(result)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

{'faithfulness': 0.8487, 'answer_relevancy': 0.4967, 'context_precision': 0.0000, 'context_recall': 0.0000}


In [5]:
# 샘플별 점수 테이블화
import pandas as pd

df = result.to_pandas()
cols = [c for c in ['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'] if c in df.columns]
df[cols]

,faithfulness,answer_relevancy,context_precision,context_recall
0,0.818182,0.504009,0.0,0.0
1,1.000000,0.610632,0.0,0.0
2,0.750000,0.539013,0.0,0.0
3,0.857143,0.387261,0.0,0.0
4,0.818182,0.442361,0.0,0.0


In [ ]:
# 지표별 평균값 시각화
import matplotlib.pyplot as plt

metric_cols = [c for c in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'] if c in df.columns]
if metric_cols:
    means = df[metric_cols].mean()
    ax = means.plot(kind='barh', figsize=(7, 3), xlim=(0, 1))
    ax.set_xlabel('score')
    ax.set_title('RAGAS metrics — capstone baseline')
    for i, v in enumerate(means):
        ax.text(v + 0.01, i, f'{v:.2f}', va='center')
    plt.tight_layout()
    plt.show()

## 정리

- RAGAS의 4개 지표는 **retriever 레이어(precision, recall)** 와 **generator 레이어(faithfulness, answer_relevancy)** 로 나뉜다. 원인 레이어를 구분해서 고쳐야 한다.
- 모든 지표는 **[0, 1]** 범위, 클수록 좋음. 실무 목표선은 보통 **0.8 이상**, 특히 faithfulness는 **0.9 이상**이 안전선.
- RAGAS는 **LLM judge 기반**이므로 judge 모델의 성능이 지표 신뢰도의 상한. 폐쇄망에서는 로컬 LLM을 judge로 지정해 완전 오프라인 평가 가능.
- **개선 순서**: `context_recall` → `context_precision` → `faithfulness` → `answer_relevancy`. 앞 단계가 무너지면 뒤 단계를 고쳐도 의미 없다.

## 더 읽어보기
- [RAGAS 공식 문서](https://docs.ragas.io/en/stable/)
- [RAGAS metrics 상세 (개념)](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/index.html)